[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulnovello/Advanced-AI/blob/main/PP2%3A%20Vision/CLIP.ipynb)

## From Classification to CLIP

In the previous notebook, we saw how a Vision Transformer (ViT) converts an image into a sequence of tokens (just like words in a sentence) and uses them to predict an image class.

We also discussed the limitations of **vanilla classification**:

- Models are restricted to a **fixed set of labels**
- Class labels provide a **weak supervision signal** (e.g., “dog” vs. a full description)
- Adding new categories requires **retraining**
- Labels do not capture semantic relationships between concepts

In particular, we examined the limitations of the supervision signal generated by discrete class labels. A single word often throws away a large amount of semantic information present in the image.

So the question becomes:

> Can we learn from richer supervision than just class IDs?

This is where **CLIP** comes in.

Instead of training a model to predict a fixed label, CLIP learns to align **images and text** in a shared embedding space.

In this notebook, we will explore what becomes possible when images and text are projected into the *same vector space*:

- 🔍 Load CLIP and understand its dual-encoder architecture  
- 🏷️ Perform zero-shot classification —> classify images with arbitrary label sets, without fine-tuning  
- 📝 Rank captions by how well they describe an image  
- 🧪 Study prompt engineering and measure how phrasing affects accuracy  
- 🎮 Bonus: build a mini image search engine using CLIP embeddings  

By the end, you’ll see how moving from fixed class labels to language supervision fundamentally changes what a vision model can do.

> 💡 **Before you start**: make sure you have run the install cell below. On Colab, this should take ~1 minute.

In [ ]:
!pip install -q transformers datasets torch torchvision matplotlib Pillow

We will also download a few test images.  
It is possible that downloading the last image will fail at first, but you can try again, it should work after a few attempts.

In [ ]:
!wget -O cat.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
!wget -O dog.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg"
!wget -O parrot.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/6/65/Amazona_albifrons_-upper_body_of_pet.jpg/1920px-Amazona_albifrons_-upper_body_of_pet.jpg"
!wget -O car.jpg "https://upload.wikimedia.org/wikipedia/commons/e/e5/Fiat_500_in_Emilia-Romagna.jpg"

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from io import BytesIO
import requests
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.manual_seed(42)

### Loading sample images

In [ ]:
def load_image(path):
    """Load a local image file and convert to RGB."""
    return Image.open(path).convert("RGB")

IMAGE_PATHS = {
    "cat":    "cat.jpg",
    "dog":    "dog.jpg",
    "parrot": "parrot.jpg",
    "car":    "car.jpg",
}

sample_images = {name: load_image(path) for name, path in IMAGE_PATHS.items()}

# Preview
n = len(sample_images)
fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
if n == 1: # Handle case with only one image
    axes = [axes]
for ax, (name, img) in zip(axes, sample_images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("Our sample images", fontsize=12)
plt.tight_layout()
plt.show()

---
## 1. Loading CLIP

We will now load the pretrained model **`openai/clip-vit-base-patch32`** from Hugging Face.

This model combines:

- A **Vision Transformer (ViT-B/32)** as the image encoder  
  - "B" = Base size  
  - "32" = 32×32 pixel patch size  
- A **Transformer-based text encoder** for processing tokenized sentences  

Both encoders project their outputs into a **shared embedding space** of fixed dimension.

![](https://images.ctfassets.net/kftzwdyauwt9/fbc4f633-9ad4-4dc2-3809c22df5e0/0bd2d5abf90d052731538613e4a42668/overview-a.svg)

After loading the model, we will:

1. Inspect its main components  
2. Understand the dual-encoder structure  
3. Examine how image and text features are produced  
4. Verify the dimensionality of the shared embedding space  

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"

# CLIPProcessor handles both image preprocessing AND text tokenization
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_img    = sum(p.numel() for p in model.vision_model.parameters())
n_txt    = sum(p.numel() for p in model.text_model.parameters())

print(f"Model: {MODEL_NAME}")
print(f"Total parameters:         {n_params:>12,}")
print(f"  Vision encoder (ViT):   {n_img:>12,}")
print(f"  Text encoder:           {n_txt:>12,}")
print(f"\nEmbedding dimension: {model.config.projection_dim}")
print(f"Max text token length:   {model.config.text_config.max_position_embeddings}")

Let's start with a quick demo on how to get one image embedding and one text embedding.  
Compute the cosine similarity between the image and the text embeddings.

In [ ]:
demo_image = list(sample_images.values())[0]
demo_text  = ["a photo of a cat", "a photo of a dog"]

inputs = processor(
    text=demo_text,
    images=demo_image,
    return_tensors="pt",
    padding=True
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

img_emb  = outputs.image_embeds   # (1, 512)
text_emb = outputs.text_embeds    # (2, 512)

print("Image embedding shape:", img_emb.shape)
print("Text embeddings shape: ", text_emb.shape)
print()
print("Cosine similarities (already L2-normalised by CLIP):")
sims = ...  # compute the cosine similarity between the image and the text embeddings, img_emb and text_emb are already normalized
for label, sim in zip(demo_text, sims):
    print(f"  '{label}': {sim.item():.4f}")
print()
print("⚠️  Raw logit scores are scaled by a learned temperature parameter.")
print(f"   model.logit_scale = {model.logit_scale.exp().item():.2f}")

### About the Logit Scale (Temperature Parameter)

You may notice that CLIP applies a **learned scaling factor** to the similarity scores:

`model.logit_scale.exp()`

This corresponds to a **temperature parameter** used in contrastive learning.

---

#### Why is scaling needed?

We compute similarities using a dot product between **L2-normalized embeddings**:

$$
\text{similarity} = \mathbf{z}_{image} \cdot \mathbf{z}_{text}
$$

Since the embeddings are normalized, this is equivalent to **cosine similarity**, and values typically lie in the range \([-1, 1]\).

However, these raw cosine similarities are often too small in magnitude to produce confident probabilities after applying a softmax.

To sharpen the distribution, CLIP multiplies the similarities by a learned scalar:

$$
\text{logits} = \tau \cdot (\mathbf{z}_{image} \cdot \mathbf{z}_{text})
$$

where:

- $\tau = \exp(\text{logit\_scale})$
- $\tau$ is learned during training
- Larger $\tau$ → sharper softmax distribution  
- Smaller $\tau$ → smoother, more uniform probabilities  


Think of this as a **confidence amplifier**:

- If similarities are close together, increasing the scale makes the best match stand out more.
- It allows the model to control how “decisive” its predictions are.
- The optimal value is learned automatically during contrastive training.

In practice, this scaling plays an important role in CLIP’s strong zero-shot performance.

Try now to add a new and complex photo and see how the model performs  with different captions.  Try with different granularity of details.

## 2. Zero-Shot Image Classification

Standard image classifiers are trained with a **fixed, predefined set of labels**.  
Once training is finished, the model can only predict among those classes.

If you want new categories, you must:
- Collect labeled data  
- Retrain (or fine-tune) the model  
- Redefine the classifier head  

CLIP works differently.

Instead of learning a fixed classification layer, CLIP learns a **shared embedding space** between images and text.

The trick is simple:

1. Encode the image → get an image embedding  
2. Encode your candidate label names as text (e.g., `"a photo of a cat"`, `"a photo of a car"`)  
3. Compute similarity between the image embedding and each text embedding  
4. Rank the labels by similarity  

The highest-similarity label becomes the prediction.

No fine-tuning.  
No retraining.  
Just new text → new classifier.

This is called **zero-shot classification**: the model can recognize categories it was never explicitly trained as fixed output classes, simply by leveraging language.

In [ ]:
def plot_zero_shot(images_dict, labels, probs, title="Zero-Shot Classification"):
    """Plot images with their predicted label distributions."""
    n = len(images_dict)
    fig, axes = plt.subplots(n, 2, figsize=(12, 3 * n),
                              gridspec_kw={"width_ratios": [1, 2.5]})
    if n == 1:
        axes = axes.reshape(1, 2)

    for row, ((name, image), prob) in enumerate(zip(images_dict.items(), probs)):
        # Image
        axes[row, 0].imshow(image.resize((224, 224)))
        axes[row, 0].set_title(name, fontsize=10, fontweight="bold")
        axes[row, 0].axis("off")

        # Sorted bar chart
        order = np.argsort(prob)[::-1]
        sorted_labels = [labels[i] for i in order]
        sorted_probs  = prob[order]
        colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(labels))]

        axes[row, 1].barh(range(len(labels)), sorted_probs[::-1], color=colors[::-1])
        axes[row, 1].set_yticks(range(len(labels)))
        axes[row, 1].set_yticklabels(sorted_labels[::-1], fontsize=9)
        axes[row, 1].set_xlim(0, 1)
        axes[row, 1].set_xlabel("Probability")
        axes[row, 1].set_title(f"Predicted: '{sorted_labels[0]}' ({sorted_probs[0]:.1%})", fontsize=10)
        for i, (b, p) in enumerate(zip(axes[row, 1].patches, sorted_probs[::-1])):
            axes[row, 1].text(b.get_width() + 0.01, b.get_y() + b.get_height() / 2,
                              f"{p:.1%}", va="center", fontsize=8)

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

Complete this function to perform zero-shot classification.  
Steps:
- Build text prompts from labels
- Encode text labels (do once, reuse for all images)
- Encode images
- Compute cosine similarity between the image and the text embeddings
- Use softmax to get the probabilities


In [ ]:
def clip_zero_shot(images, labels, model, processor, prompt_template="a photo of a {}"):
    """
    Zero-shot classify a list of images against a set of text labels.

    Returns: probs array of shape (n_images, n_labels)
    """
    all_probs = []
    with torch.no_grad():
        ...
    return np.array(all_probs)  # (n_images, n_labels)

# ── Run zero-shot classification with a basic animal + food label set ─────────

labels_basic = ["cat", "dog", "pizza", "sushi", "ice cream", "bird", "car"]

images_list = list(sample_images.values())
probs = clip_zero_shot(images_list, labels_basic, model, processor)

plot_zero_shot(sample_images, labels_basic, probs,
               title=f"Zero-Shot CLIP — labels: {labels_basic}")


---
## 3. Image–Text Similarity Scoring

Classification forces a single winner.

CLIP is more flexible: it produces **continuous similarity scores** between *any* image and *any* text.  
That means we can do much more than “pick a label”:

- 📝 **Rank captions** by how well they match an image  
- 🚫 **Detect mismatches** (low similarity = likely wrong description)  
- 📏 **Score descriptions** on a smooth scale instead of a hard yes/no decision  

In this section, we will:

1. Encode a set of images and a set of candidate captions  
2. Compute all pairwise similarities  
3. Visualize the result as a **similarity matrix** (image × caption)

In [ ]:
def clip_similarity_matrix(images_dict, captions, model, processor):
    """
    Compute cosine similarity between every image and every caption.
    Returns: matrix of shape (n_images, n_captions), values in [-1, 1]
    """
    ...

    sim_matrix = ...  # (n_images, n_captions)
    return sim_matrix


def plot_similarity_matrix(images_dict, captions, sim_matrix):
    """Visualize the image×caption similarity matrix."""
    image_names = list(images_dict.keys())
    n_img = len(image_names)
    n_cap = len(captions)

    # Short captions for axis labels
    short_caps = [c[:40] + "…" if len(c) > 40 else c for c in captions]

    fig = plt.figure(figsize=(max(10, n_cap * 1.5), 3 + n_img * 1.5))
    gs = gridspec.GridSpec(n_img, n_img + n_cap,
                            wspace=0.05, hspace=0.1)

    # Left: image thumbnails
    for i, (name, img) in enumerate(images_dict.items()):
        ax = fig.add_subplot(gs[i, :n_img])
        ax.imshow(img.resize((112, 112)))
        ax.set_ylabel(name, fontsize=9, rotation=0, labelpad=40, va="center")
        ax.set_xticks([])
        ax.set_yticks([])

    # Right: heatmap
    ax_heat = fig.add_subplot(gs[:, n_img:])
    im = ax_heat.imshow(sim_matrix, cmap="RdYlGn",
                         vmin=sim_matrix.min(), vmax=sim_matrix.max(),
                         aspect="auto")
    ax_heat.set_xticks(range(n_cap))
    ax_heat.set_xticklabels(short_caps, rotation=35, ha="right", fontsize=8)
    ax_heat.set_yticks(range(n_img))
    ax_heat.set_yticklabels(image_names, fontsize=9)
    ax_heat.set_title("Image × Caption Cosine Similarity", fontsize=11)
    plt.colorbar(im, ax=ax_heat, fraction=0.03)

    for i in range(n_img):
        for j in range(n_cap):
            ax_heat.text(j, i, f"{sim_matrix[i, j]:.2f}",
                         ha="center", va="center", fontsize=8,
                         color="black" if 0.3 < sim_matrix[i, j] < 0.7 else "white")

    plt.tight_layout()
    plt.show()


# Captions: some correct, some plausible but wrong, some clearly wrong
CAPTIONS = [
    "a domestic cat sitting",
    "a golden retriever dog",
    "a delicious pizza with toppings",
    "a bike in the city of London",
    "a tiger on the grass",
    "an animal running outside",
    "a red car on the street",
    "a yellow car on the beach",
]

sim_matrix = clip_similarity_matrix(sample_images, CAPTIONS, model, processor)
plot_similarity_matrix(sample_images, CAPTIONS, sim_matrix)


Try to fool clip with some close but uncorrect captions.

In [ ]:
# Try your own captions
MY_CAPTIONS = [
    "a cat jumping",
    "a cat waiting",
    "a little dog",
    "a yellow dog",
    "a dog eating an animal",
    "a green kiwi",
    "a red toy",
    "a city picture",
]

my_sim = clip_similarity_matrix(sample_images, MY_CAPTIONS, model, processor)
plot_similarity_matrix(sample_images, MY_CAPTIONS, my_sim)

---
## 4. Prompt Engineering

One of the key findings of the original CLIP paper is that **how you phrase a label matters**.

Because CLIP aligns images with *natural language*, different textual formulations of the same concept can produce noticeably different embeddings — and therefore different classification accuracy.

For example:

- `"dog"`  
- `"a photo of a dog"`  
- `"a high-quality photo of a dog, a type of pet"`  

All three refer to the same concept, but they provide different amounts of context and structure.  
CLIP was trained on image–caption pairs, so prompts that resemble natural captions often perform better than single-word labels.

In this section, we will:

1. Define a set of prompt templates  
2. Generate label descriptions using each template  
3. Perform zero-shot classification with each formulation  
4. Compare accuracies quantitatively  

This systematic evaluation will show how **prompt engineering** can significantly influence performance — even without changing the model.

Use the ```clip_zero_shot```method to assess which of these templates gives the best results.





In [ ]:
TRUE_LABELS = list(sample_images.keys())
PROMPT_TEMPLATES = [
    "{}",                                              # bare label
    "a photo of a {}",                                 # CLIP default
    "a photograph of a {}",
    "a high quality photo of a {}",
    "a photo of the {}",                               # definite article
    "this is a {}",
    "an image showing a {}",
    "a close-up photo of a {}",
    "a blurry photo of a {}",                          # negative — should hurt!
    "a drawing of a {}",                               # wrong modality
]


target_name = TRUE_LABELS[0]  # first image (cat)
target_image = list(sample_images.values())[0]
target_label = TRUE_LABELS[0]

confidence_per_template = []
for template in PROMPT_TEMPLATES:
    probs = clip_zero_shot([target_image], TRUE_LABELS, model, processor,
                             prompt_template=template)[0]
    correct_idx = TRUE_LABELS.index(target_label)
    confidence_per_template.append(probs[correct_idx])

ranked = np.argsort(confidence_per_template)[::-1]

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#2ecc71" if i == ranked[0] else
          "#e74c3c" if i == ranked[-1] else "#3498db"
          for i in range(len(PROMPT_TEMPLATES))]
bars = ax.bar(
    range(len(PROMPT_TEMPLATES)),
    [confidence_per_template[i] for i in ranked],
    color=[colors[i] for i in ranked]
)
ax.set_xticks(range(len(PROMPT_TEMPLATES)))
ax.set_xticklabels([f'"{PROMPT_TEMPLATES[i]}"' for i in ranked],
                    rotation=30, ha="right", fontsize=8)
ax.set_ylabel("P(correct label)")
ax.set_title(f"Confidence in correct label ('{target_label}') per prompt template", fontsize=11)
ax.set_ylim(0, 1)
for bar, conf in zip(bars, [confidence_per_template[i] for i in ranked]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f"{conf:.1%}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

print("\n💡 Takeaway: phrasing alone can swing confidence by 10–30 percentage points.")
print("   CLIP's original paper used 80 different prompt templates and averaged them.")

Takeaway: phrasing alone can swing confidence by 10–30 percentage points. It is not the case here because our labels are clearly separated and we only have a few of them but it might be more significant with more classes.
CLIP's original paper used 80 different prompt templates and averaged them.

---
## 5. Mini Image Search Engine

One of the most powerful consequences of a shared image–text embedding space is **natural language search**.

With CLIP, searching for images becomes extremely simple:

1. Encode a text query (e.g., `"a superhero flying in the sky"`)
2. Encode a set of images
3. Compute cosine similarity
4. Retrieve the images with the highest similarity scores

No metadata.  
No manual tags.  
No supervised classifier.

Just embeddings and similarity.

In this section, we’ll build a tiny image search engine over a collection of **20 movie posters**.  
Given a text query, the system will rank the posters according to semantic similarity and return the most relevant ones.

This demonstrates how CLIP turns search into a pure vector retrieval problem — powered entirely by multimodal alignment.

In [ ]:
from datasets import load_dataset

ds = load_dataset("GentextS6/10k_movie_posters")

Before we can search, we need to **build an index of image embeddings**.

In this cell, we:

1. Select a subset of the dataset (`INDEX_SIZE = 500`)
2. Iterate through the images
3. Clean and store valid images in a dictionary
4. Assign each image a unique title (since the dataset does not provide one directly)
5. Skip corrupted or missing images safely
6. Preview the first 20 indexed posters in a grid

In [ ]:
from tqdm import tqdm
print("Building image index (streaming 10k_movie_posters, 1000 images)...")

INDEX_SIZE = 1000

index_images = {}   # title → PIL Image
index_titles = []   # ordered list of titles


# Get the list of image objects from the dataset
images_data = ds['train']['image']

# Iterate over the images, limiting to INDEX_SIZE
for i in tqdm(range(INDEX_SIZE)):
    title = f"poster_{i:03d}"
    # Ensure unique keys
    key = title if title not in index_images else f"{title}_{i}"

    try:
        img = images_data[i] # Access image by index
        if not isinstance(img, type(None)):
            index_images[key] = img.convert("RGB")
            index_titles.append(key)
        else:
            print(f"Skipping image at index {i} due to it being None.")
    except Exception as e: # Catch UnidentifiedImageError and other potential issues
        print(f"Skipping image at index {i} due to error: {type(e).__name__} - {e}")

print(f"Indexed {len(index_images)} movie posters.")

# Show a grid preview (first 20)
preview = list(index_images.items())[:20]
cols = 5
rows = (len(preview) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 3))
for ax, (name, img) in zip(axes.flatten(), preview):
    ax.imshow(img.resize((112, 150)))
    ax.set_title(name, fontsize=6, wrap=True)
    ax.axis("off")
for ax in axes.flatten()[len(preview):]:
    ax.axis("off")
plt.suptitle(f"Movie Poster Index — first {len(preview)} of {INDEX_SIZE}", fontsize=12)
plt.tight_layout()
plt.show()

Now that we have collected and cleaned our images, we convert them into **CLIP embeddings**.

This step transforms raw pixels into vectors in the shared image–text embedding space.

---

#### What Happens in This Cell

1. Extract image names and image objects from the index  
2. Use the CLIP `processor` to prepare the images as model inputs  
3. Run the image encoder (`model.get_image_features`)  
4. Obtain a matrix of embeddings of shape:

$$
n_{images, embedding dim}
$$

For our setup:

- `n_images = 500`
- `embedding_dim = 512`

---

#### Why We Normalize

After obtaining the embeddings, we apply L2 normalization:

$$
\mathbf{z} \leftarrow \frac{\mathbf{z}}{\|\mathbf{z}\|}
$$

This ensures:

- All embeddings lie on the unit hypersphere  
- Dot product = cosine similarity  
- Similarity comparisons are scale-invariant  

This is essential because CLIP was trained using cosine similarity in a contrastive objective.

---

At this point, we have a **fixed embedding matrix** representing our entire image database.  
Searching will now reduce to a fast vector similarity computation.

In [ ]:
index_names  = list(index_images.keys())
index_imgs   = list(index_images.values())

img_inputs = processor(images=index_imgs, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    raw_index_embeddings = model.get_image_features(**img_inputs)  # (500, 512)
    index_embeddings = raw_index_embeddings.pooler_output
    index_embeddings = index_embeddings / index_embeddings.norm(dim=-1, keepdim=True)

print(f"Index embedding matrix: {index_embeddings.shape}  (n_images × embedding_dim)")
print("✅ Index ready")


Now we can turn CLIP into a tiny **natural language image search engine**.

Complete the function below which implements the core retrieval pipeline:

1. **Encode the text query** into a CLIP text embedding  
2. **Normalize** the embedding so dot products correspond to cosine similarity  
3. Compute similarity between the query embedding and **all indexed image embeddings**  
4. Return the **top-k** most similar posters  
5. Visualize the results in a row (highlighting the best match)

In [ ]:
def search(query_text, index_embeddings, index_names, index_images,
           model, processor, top_k=5):
    """
    Natural language image search.
    Returns and plots the top_k most similar images for a text query.
    """
    # Encode query text
    text_inputs = processor(text=[query_text], return_tensors="pt",
                              padding=True).to(device)
    with torch.no_grad():
        raw_query_emb = model.get_text_features(**text_inputs)  # (1, 512)
        if hasattr(raw_query_emb, 'pooler_output'):
            query_emb = raw_query_emb.pooler_output
        else:
            query_emb = raw_query_emb
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)

    # Cosine similarities against all index images
    sims = (query_emb @ index_embeddings.T)[0].cpu().numpy()  # (n_images,)
    top_indices = np.argsort(sims)[::-1][:top_k]

    # Plot results
    fig, axes = plt.subplots(1, top_k, figsize=(top_k * 2.8, 3))
    for rank, (ax, idx) in enumerate(zip(axes, top_indices)):
        ax.imshow(index_images[index_names[idx]].resize((150, 150)))
        ax.set_title(f"#{rank+1}  {index_names[idx]}\nsim={sims[idx]:.3f}",
                     fontsize=8)
        ax.axis("off")
        if rank == 0:
            for spine in ax.spines.values():
                spine.set_edgecolor("#2ecc71")
                spine.set_linewidth(3)
    plt.suptitle(f'Search: "{query_text}"', fontsize=12)
    plt.tight_layout()
    plt.show()

    return [(index_names[i], float(sims[i])) for i in top_indices]


# Try some queries
for query in [
    "a dark thriller movie",
    "animated family film",
    "space science fiction",
]:
    search(query, index_embeddings, index_names, index_images, model, processor)


You can try with your own queries.

In [ ]:
query = "a superhero action movie"  # ← change me!

results = search(query, index_embeddings, index_names,
                  index_images, model, processor)
print("\nTop results:")
for name, sim in results:
    print(f"  {sim:.4f}  {name}")



So far, we searched images using **text queries**.  
Now we’ll perform **image-to-image retrieval**.

Given a query poster, we want to retrieve the most similar posters from our index.

But there are different ways to define “similar”:

- **Semantic similarity** — What is the movie about? (genre, themes, mood, characters)
- **Visual similarity** — What does the poster look like? (colors, layout, textures, composition)

To illustrate this difference, we compare two embedding models:

- **CLIP** (semantic): trained with language supervision to align images with captions  
- **VGG16** (visual): a convolutional network trained for ImageNet classification (no text supervision)

---

### What We Will Do

We implement an image-to-image retrieval pipeline with both models:

1. **Load VGG16** and use it as a feature extractor  
   - We extract features from the penultimate classifier layer to obtain a **4096-dimensional vector**
2. **Pre-compute VGG embeddings** for all indexed posters  
   - Just like in a real search engine, this is done once and reused
3. For a given **query image**, compute:
   - A CLIP image embedding → compare against CLIP index embeddings  
   - A VGG image embedding → compare against VGG index embeddings  
4. Display the **top-k results side by side**
   - Row 1: CLIP results (semantic retrieval)  
   - Row 2: VGG16 results (visual retrieval)

---

### What to Observe

When you run the comparison, look carefully at the differences:

- Does **CLIP** retrieve posters that match the *concept* or *theme* of the query?
- Does **VGG16** retrieve posters that share visual characteristics like color palette or layout?

This experiment makes the distinction clear:
CLIP often captures *meaning*, while VGG primarily captures *appearance*.

In [ ]:
import torchvision.models as tv_models
import torchvision.transforms as T

# VGG feature extractor (penultimate layer, 4096-d)
print("Loading VGG16 backbone...")
vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.DEFAULT).to(device)
vgg.eval()

# vgg pre-processing function
vgg_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def vgg_embed(pil_image):
    """Return a L2-normalised 4096-d VGG feature vector."""
    x = vgg_transform(pil_image).unsqueeze(0).to(device)  # (1, 3, 224, 224)
    with torch.no_grad():
        feats = vgg.features(x)                            # (1, 512, 7, 7)
        feats = vgg.avgpool(feats)                         # (1, 512, 7, 7)
        feats = torch.flatten(feats, 1)                    # (1, 25088)
        feats = vgg.classifier[:5](feats)                  # (1, 4096) — after 2nd ReLU
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.squeeze(0)                                # (4096,)


# Pre-compute VGG embeddings for the index
print("Pre-computing VGG embeddings for all index images...")
vgg_index_embeddings = torch.stack(
    [vgg_embed(img) for img in index_images.values()]
)  # (INDEX_SIZE, 4096)
print(f"VGG index: {vgg_index_embeddings.shape}")


# Dual search function
def image_search_compare(query_image, query_name,
                          clip_index_embeddings, vgg_index_embeddings,
                          index_names, index_images,
                          model, processor, top_k=5):
    """
    Search the index with a query image using BOTH CLIP and VGG,
    then display results side by side for comparison.
    """
    # ── CLIP query embedding ──
    img_input = processor(images=query_image, return_tensors="pt").to(device)
    with torch.no_grad():
        raw_clip_q = model.get_image_features(**img_input)
        if hasattr(raw_clip_q, 'pooler_output'):
            clip_q = raw_clip_q.pooler_output
        else:
            clip_q = raw_clip_q
        clip_q = clip_q / clip_q.norm(dim=-1, keepdim=True)

    clip_sims = (clip_q @ clip_index_embeddings.T)[0].cpu().numpy()

    # Remove the query image from results
    query_idx = index_names.index(query_name)
    clip_sims[query_idx] = -np.inf
    clip_top = np.argsort(clip_sims)[::-1][:top_k]

    # ── VGG query embedding ──
    vgg_q    = vgg_embed(query_image).unsqueeze(0)          # (1, 4096)
    vgg_sims = (vgg_q @ vgg_index_embeddings.T)[0].cpu().numpy()

    # Remove the query image from results
    vgg_sims[query_idx] = -np.inf
    vgg_top = np.argsort(vgg_sims)[::-1][:top_k]

    # ── Plot ──
    fig, axes = plt.subplots(3, top_k + 1,
                              figsize=((top_k + 1) * 2.5, 9))

    # Row 0: query image (centered)
    for ax in axes[0]:
        ax.axis("off")
    axes[0][0].imshow(query_image.resize((150, 200)))
    axes[0][0].set_title(f"Query:\n{query_name}", fontsize=9, fontweight="bold")
    axes[0][0].axis("on")
    for spine in axes[0][0].spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(2)

    # Row 1: CLIP results
    axes[1][0].text(0.5, 0.5, "CLIP"
"(semantic)", ha="center", va="center",
                    fontsize=11, fontweight="bold", color="#2980b9",
                    transform=axes[1][0].transAxes)
    axes[1][0].axis("off")
    for rank, (ax, idx) in enumerate(zip(axes[1][1:], clip_top)):
        ax.imshow(index_images[index_names[idx]].resize((150, 200)))
        ax.set_title(f"#{rank+1}\n{index_names[idx][:20]}\nsim={clip_sims[idx]:.3f}",
                     fontsize=7)
        ax.axis("off")
        if rank == 0:
            for spine in ax.spines.values():
                spine.set_edgecolor("#2ecc71")
                spine.set_linewidth(3)

    # Row 2: VGG results
    axes[2][0].text(0.5, 0.5, "VGG16"
"(visual)", ha="center", va="center",
                    fontsize=11, fontweight="bold", color="#c0392b",
                    transform=axes[2][0].transAxes)
    axes[2][0].axis("off")
    for rank, (ax, idx) in enumerate(zip(axes[2][1:], vgg_top)):
        ax.imshow(index_images[index_names[idx]].resize((150, 200)))
        ax.set_title(f"#{rank+1}\n{index_names[idx][:20]}\nsim={vgg_sims[idx]:.3f}",
                     fontsize=7)
        ax.axis("off")
        if rank == 0:
            for spine in ax.spines.values():
                spine.set_edgecolor("#e74c3c")
                spine.set_linewidth(3)

    plt.suptitle(f"Image-to-Image Search — CLIP (blue) vs VGG16 (red)\n"
                 f"Query: \"{query_name}\"", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()


# Comparison on a few random query posters
import random

index_names_list = list(index_images.keys())
n = 5  # number of random query posters

random_indices = random.sample(range(len(index_names_list)), n)

for idx in random_indices:
    name = index_names_list[idx]
    img  = index_images[name]

    image_search_compare(
        img, name,
        index_embeddings, vgg_index_embeddings,
        index_names_list, index_images,
        model, processor,
        top_k=5,
    )

---
## 6. CLIP's Weaknesses

CLIP is powerful, but it is not perfect.

Because it is trained with large-scale image–text alignment (not explicit reasoning supervision), it can struggle with certain types of queries.

We will now explore a few common failure modes:

### 🔢 Counting

CLIP is not explicitly trained to count objects.  
Queries like:

- "six circles"
- "three red squares"

may fail because the model focuses more on *object presence* than precise numerosity.

It often captures *what is in the image*, but not *how many*.

---

### ↔️ Spatial Relationships

CLIP can struggle with fine-grained spatial reasoning:

- "a square on the left"
- "a square at the top"
- "a circle below a square"

While the model encodes global image features, it does not have an explicit mechanism for precise spatial logic.  
Subtle positional differences may not strongly affect the embedding.

---

### 🚫 Negation

Negation is another challenge:

- "a photo without a dog"
- "a poster with no text"

Models trained contrastively often encode what *is present* more strongly than what *is absent*.  
Negation requires compositional reasoning that is not directly enforced during training.

---

These examples highlight an important idea:

CLIP learns powerful **semantic alignment**,  
but it is not a symbolic reasoning system.

Understanding these limitations helps us better interpret both its strengths and its weaknesses.

In [ ]:
import random
from PIL import Image, ImageDraw
import math

def generate_circles_image(num_circles, size=(224, 224), background_color=(255, 255, 255)):
    """Generates a fake PIL image containing multiple random circles of the same size without overlapping."""
    img = Image.new('RGB', size, background_color)
    draw = ImageDraw.Draw(img)

    width, height = size

    # Adjusted radius calculation to make circles smaller and fit more
    fixed_radius = min(width, height) // (num_circles * 3)  # Increased divisor for smaller circles
    if fixed_radius < 5:
        fixed_radius = 5  # Minimum radius

    placed_circles = []  # Store (x, y, radius) of placed circles

    for _ in range(num_circles):
        placed = False
        attempts = 0
        max_attempts = 100  # Prevent infinite loops if placement is impossible

        while not placed and attempts < max_attempts:
            attempts += 1

            # Random position, ensuring circle is mostly within bounds
            x = random.randint(fixed_radius, width - fixed_radius)
            y = random.randint(fixed_radius, height - fixed_radius)

            overlaps = False
            for px, py, pr in placed_circles:
                distance = math.sqrt((x - px)**2 + (y - py)**2)
                if distance < (fixed_radius + pr):
                    overlaps = True
                    break

            if not overlaps:
                placed_circles.append((x, y, fixed_radius))
                placed = True

        if not placed:
            print(f"Warning: Could not place all circles without overlap. Placed {len(placed_circles)} out of {num_circles}.")
            break

    for x, y, radius in placed_circles:
        fill_color = (random.randint(0, 255),
                      random.randint(0, 255),
                      random.randint(0, 255))
        draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=fill_color)

    return img


def generate_single_square(position="left", size=(224, 224), background_color=(255, 255, 255),
                          square_size=50, margin=20, color=(0, 0, 0)):
    """
    Generates an image with a single filled square placed at a specific position.
    position: "left" or "top"
    """
    img = Image.new("RGB", size, background_color)
    draw = ImageDraw.Draw(img)
    W, H = size

    if position == "left":
        # center vertically, near left edge
        x0 = margin
        y0 = (H - square_size) // 2
    elif position == "top":
        # center horizontally, near top edge
        x0 = (W - square_size) // 2
        y0 = margin
    else:
        raise ValueError("position must be 'left' or 'top'")

    x1 = x0 + square_size
    y1 = y0 + square_size
    draw.rectangle([x0, y0, x1, y1], fill=color)

    return img


print("Generating and adding synthetic failure-mode images...")

# 1) Counting image: multiple circles
multi_circles_image = generate_circles_image(num_circles=6, size=(224, 224), background_color=(255, 255, 255))
sample_images["multi_circles"] = multi_circles_image

# 2) Spatial images: single square on the left / on the top
square_left_image = generate_single_square(position="left", size=(224, 224), square_size=50, margin=20, color=(0, 0, 0))
square_top_image  = generate_single_square(position="top",  size=(224, 224), square_size=50, margin=20, color=(0, 0, 0))

sample_images["square_left"] = square_left_image
sample_images["square_top"]  = square_top_image

sample_images.pop("dog");
sample_images.pop("parrot");
sample_images.pop("car");

print(f"Updated sample_images: {list(sample_images.keys())}")

# Preview updated sample_images
n = len(sample_images)
fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
if n == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, sample_images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("Our sample images (updated)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
failure_tests = [
    {
        "name": "Counting",
        "desc": "CLIP struggles to distinguish quantities",
        "labels": ["one circle", "two circles", "three circles", "four circles", "five circles", "six circles", "many circles"],
        "images": {"multi_circles": sample_images["multi_circles"]},
    },
    {
        "name": "Spatial reasoning",
        "desc": "CLIP doesn't reliably understand left/right/above",
        "labels": ["square on the left", "square on the right",
                   "square in the center", "square at the top"],
        "images": {k: v for k, v in sample_images.items() if "square" in k}, # FIX: Iterate over all items to find 'square' images
    },
    {
        "name": "Negation",
        "desc": "CLIP often ignores 'not' in text",
        "labels": ["a photo of a bird", "a photo of a dog",
                   "a photo that is NOT a cat", "a photo that is NOT a dog"],
        "images": {"cat": sample_images["cat"]},
    },
]

for test in failure_tests:
    probs = clip_zero_shot(
        list(test["images"].values()),
        test["labels"], model, processor,
        prompt_template="{}"  # no template wrapper — labels are full sentences
    )
    plot_zero_shot(
        test["images"], test["labels"], probs,
        title=f"⚠️  Failure Mode: {test['name']} — {test['desc']}"
    )

---
## ✅ Summary — What We Learned

| Concept | Key Insight |
|---|---|
| **Contrastive training** | CLIP trains by matching image and text embeddings — no class labels needed |
| **Shared embedding space** | Any image and any text can be compared with cosine similarity |
| **Zero-shot classification** | Encode label names as text → instant classifier for any label set |
| **Prompt engineering** | Phrasing matters: `"a photo of a {}"` consistently outperforms bare labels |
| **Image search** | Text queries find images; image queries find similar images — same operation |
| **CLIP's limits** | Counting, spatial reasoning, negation — things that need compositional understanding |

---
### 🚀 Up Next:  VLMs

CLIP can score similarity between images and text but it can't **generate** text or **reason** about images.  
The next step: attach CLIP's visual encoder to a language model → **Vision-Language Models**.